# Initializing the environment

In [ ]:
from gymnasium import make
from samples.llm_interface import OGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
os.environ["OPENAI_API_KEY"] = ""

MAX_EPISODES = 5
n_player1 = 2
n_player2 = 2
# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
all_classes = ['halfling_rogue.yml', 'high_elf_fighter.yml']
all_players = ["Alysha", "Bernard", "Cedric", "Didier", "Eric", "Francois", "Gertrude", "Heloise", "Isabelle"]
players = random.sample(all_players, n_player1 + n_player2)
a_player = [(random.choice(all_classes), player) for player in players[:n_player1]]
e_player = [(random.choice(all_classes), player) for player in players[n_player1:]]
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=a_player,
    enemies=e_player,
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000))




agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agents[character.name] = (OGPT4Interfacer(debug=False, explain=True, name=character.name), gr, character)

agents


Alysha rolled initiative d20(11) + 5 value 16.2
Bernard rolled initiative d20(10) + 5 value 15.2
Didier rolled initiative d20(16) + 5 value 21.2
Gertrude rolled initiative d20(2) + 5 value 7.2
Alysha rolled initiative d20(5) + 5 value 10.2
Bernard rolled initiative d20(2) + 5 value 7.2
Didier rolled initiative d20(15) + 5 value 20.2
Gertrude rolled initiative d20(12) + 5 value 17.2
Combat begins with 4 players.
Players: <p>Alysha (fighter-2) Team a</p>
<p>Bernard (rogue-2) Team a</p>
<p>Didier (fighter-2) Team b</p>
<p>Gertrude (fighter-2) Team b</p>
======== Didier starts their turn. ========
======== Didier starts their turn. ========


/Users/malek/miniconda3/envs/natural_20/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/malek/miniconda3/envs/natural_20/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(
/Users/malek/miniconda3/envs/natural_20/lib/python3.12/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


{'Didier': (<samples.llm_interface.OGPT4Interfacer at 0x14fabc2f0>,
  'b',
  Didier),
 'Gertrude': (<samples.llm_interface.OGPT4Interfacer at 0x148484dd0>,
  'b',
  Gertrude),
 'Alysha': (<samples.llm_interface.OGPT4Interfacer at 0x14fde0f80>,
  'a',
  Alysha),
 'Bernard': (<samples.llm_interface.OGPT4Interfacer at 0x168312c00>,
  'a',
  Bernard)}

In [2]:
class Experiment():
    def __init__(self, environment, dnd_environment, agents, debug=False):
        self.env = environment
        self.dnd_environment = dnd_environment
        self.agents = agents
        self.debug = debug
        self.backlog = []
        self.conversations = []
    
    def get_obs_inf(self, player):
        p_observation = self.dnd_environment.generate_observation(player)
        p_available_moves = compute_available_moves(self.dnd_environment.session, self.dnd_environment.map, player, self.dnd_environment.battle, self.dnd_environment.weapon_mappings, self.dnd_environment.spell_mappings)
        p_info = self.dnd_environment._info(p_available_moves, player)
        return p_observation, p_info
    
    def update_all_agents(self, agents_name, sender, content):
        for name in agents_name:
            self.agents[name][0].register_conversation(sender, content)
        self.conversations[-1].append((sender, content))

    def initiate_conversation(self, agents_name):
        for name in agents_name:
            self.agents[name][0].initiate_conversation()
        self.conversations.append([])

    def close_conversation(self, agents_name):
        for name in agents_name:
            obs, inf = self.get_obs_inf(self.agents[name][2])
            self.agents[name][0].close_conversation(obs, inf, self.dnd_environment.players)
        self.conversations[-1].append((None, "Conversation closed"))

    def run_conversation(self, sender, content):
        sender_gr = self.agents[sender][1]
        agent_in_the_conv = []
        for name, (_, gr, _) in self.agents.items():
            if sender_gr == gr and name != sender:
                agent_in_the_conv.append(name)
        agent_in_the_conv.append(sender)
        self.initiate_conversation(agent_in_the_conv)
        self.update_all_agents(agent_in_the_conv, sender, content)
        conv_alive = True
        conv_step = 0
        while conv_alive:
            conv_step += 1
            conv_alive = False
            for name in agent_in_the_conv:
                obs, inf = self.get_obs_inf(self.agents[name][2])
                action, descrition,  content = self.agents[name][0].select_action_for_state(obs, inf, self.dnd_environment.players, is_conversation=True)
                if action == -2:
                    self.update_all_agents(agent_in_the_conv, name, content)
                    conv_alive = True
                elif action != -3:
                    raise ValueError(f"A non conversation action {action} was used during a conversation by agent {name}")
        self.close_conversation(agent_in_the_conv)
    
    def step(self):
        current_agent, current_group, current_character = self.agents[self.dnd_environment.battle.current_turn().name]
        obs, inf = self.get_obs_inf(current_character)
        action, descrition, content = current_agent.select_action_for_state(obs, inf, self.dnd_environment.players)
        self.backlog.append((current_character.name, action, descrition))
        if action != -1:
            _, _, terminal, _, _ = self.env.step(action)
        else :
            self.backlog.append((current_character.name, -1, len(self.conversations)))
            self.run_conversation(sender=current_character.name, content=content)
            terminal = False
        return terminal
    
    def run_till_end(self, max_step= 30):
        done = False
        step = 0
        while not done and step < max_step:
            if self.debug: print(f"\n\n________________________________________________________________________________\n Starting step {step}:\n")
            done = self.step()
            step += 1

In [3]:
expe = Experiment(env, env.env.env, agents, debug=True)
expe.run_till_end(max_step=1000)



________________________________________________________________________________
 Starting step 0:

{'action_id': 2, 'description': 'attack Bernard with ranged weapon: longbow', 'explanation': "Bernard is visible and within range. As a fighter, my best use of the opening round is to try to deal damage immediately. I will use my action to attack Bernard with my longbow. I'll coordinate with Gertrude after combat begins if the situation changes."}
Didier attacked Alysha with Longbow and hits with attack roll d20(20) + 7 = 27 (critical hit)..
Alysha took d8(3 + 3) + 5 = 11 piercing damage. 


________________________________________________________________________________
 Starting step 1:



/Users/malek/miniconda3/envs/natural_20/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/malek/miniconda3/envs/natural_20/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(


{'action_id': 1, 'description': 'Move 5ft up and to the left.', 'explanation': 'Moving up and to the left will start to close the distance with Bernard, putting pressure on him and potentially letting me use melee or ranged attacks more effectively in coming rounds. It also allows me to reposition for cover or to better support Gertrude if needed.'}
Didier moved to [2, 5] 5 feet


________________________________________________________________________________
 Starting step 2:

{'action_id': 10, 'description': 'Use Second Wind as a bonus action to regain hit points.', 'explanation': 'Since I am currently at full health (24/24), using Second Wind would be a waste. I should consider repositioning or communicating with my allies instead. Instead, I will end my turn as I have no available actions left and no urgent reason to move or communicate this round.'}
Didier uses second wind to recover d10(3) + 2=5 hit points.


______________________________________________________________________

In [4]:
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


expe.backlog
metrics = combat_metrics(expe.dnd_environment)
print(metrics)
score = combat_score(metrics)
print(score)

{'win': False, 'turns_taken': 27, 'survivors': {'Bernard': (16, 16), 'Didier': (18, 24), 'Gertrude': (20, 24)}}
-1.1666666666666643


In [5]:
env.env.env.players

[('a', 'H', Alysha, [10, 4]),
 ('a', 'H', Bernard, [1, 2]),
 ('b', 'E', Didier, [3, 6]),
 ('b', 'E', Gertrude, [9, 3])]

In [6]:
agents["Didier"][0].summary
# agents

'Combat is underway. Bernard is still the primary threat, as he remains at full health, and I continue to prioritize him using my longbow from my current position, maintaining line of sight. This round, I have already used my action to attack him and see no advantage in repositioning or using other actions at this time, so I ended my turn to conserve resources. I remain alert to Bernard’s next move and the general flow of battle. Alysha is down, but I will continue to monitor Gertrude’s position and support as needed. The plan is to keep pressure on Bernard while staying flexible for any changes in the situation.'

In [7]:
# Select an action based on the initial state
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]

action = current_agent.select_action_for_state(observation, info, env.env.env.players)
print(f"Selected action: {action}")
# terminal = False
# episode = 0
# while not terminal and episode < MAX_EPISODES:
#     episode += 1
#     observation, reward, terminal, truncated, info = env.step(action)
#     if not terminal and not truncated:
#         print(env.render())
#         current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]
#         action = current_agent.select_action_for_state(observation, info)
#         print(f"Selected action: {action}")

#     if terminal or truncated:
#         print(f"Reward: {reward}")
#         break
# action

{'action_id': 13, 'description': 'second wind action', 'explanation': "I'm currently at 0 HP and unconscious as Alysha, so I can't take any actions, including using Second Wind. Therefore, I am effectively unable to act until stabilized or healed."}
Selected action: ((8, (-1, -1), (0, 0), 0, 0), 'second wind action', None)


In [8]:
info

{'available_moves': [(0, (0, 0), (7, -2), 13, 1),
  (0, (0, 0), (-2, -4), 13, 1),
  (15, (-1, -1), (0, 0), 0, 0),
  (4, (-1, -1), (0, 0), 0, 0),
  (2, (-1, -1), (0, 0), 0, 0),
  (3, (-1, -1), (0, 0), 0, 0),
  (1, (-1, -1), (0, 0), 0, 0),
  (1, (-1, 0), (0, 0), 0, 0),
  (1, (0, -1), (0, 0), 0, 0),
  (1, (1, -1), (0, 0), 0, 0),
  (1, (1, 0), (0, 0), 0, 0),
  (10, (-1, -1), (0, 0), 0, 0),
  (8, (-1, -1), (0, 0), 0, 0),
  (14, (-1, -1), (0, 0), 0, 0),
  (17, (-1, -1), (0, 0), 0, 0),
  (16, (-1, -1), (0, 0), 0, 0),
  (-1, (0, 0), (0, 0), 0, 0)],
 'current_index': 0,
 'group': 'b',
 'round': 0,
 'health': 24,
 'max_health': 24,
 'weapon_mappings': {'unarmed': 0,
  'battleaxe': 1,
  'dagger': 2,
  'quarterstaff': 3,
  'sling': 4,
  'dart': 5,
  'greatclub': 6,
  'hand_crossbow': 7,
  'handaxe': 8,
  'javelin': 9,
  'heavy_crossbow': 10,
  'light_crossbow': 11,
  'light_hammer': 12,
  'longbow': 13,
  'longsword': 14,
  'rapier': 15,
  'scimitar': 16,
  'shortsword': 17,
  'shortbow': 18,
  's

In [9]:
observation.keys()

dict_keys(['map', 'turn_info', 'conditions', 'health_pct', 'player_equipped', 'ally_name', 'enemy_name', 'ally_reactions', 'health_ally', 'health_enemy', 'ally_conditions', 'enemy_conditions', 'enemy_reactions', 'player_ac', 'ally_ac', 'enemy_ac', 'ability_info', 'player_type', 'ally_type', 'enemy_type', 'spell_slots', 'movement', 'is_reaction'])

In [10]:
observation["health_enemy"]

array([1., 1.])